# Day 042 — Exercise 3: filter_orders

**What you'll build:** `filter_orders(conn, region=None, category=None, min_revenue=None) -> list[dict]` — build a dynamic WHERE clause from optional keyword arguments using parameterized queries.

**Why it matters:** Real queries often need optional filters. Building the WHERE clause dynamically — appending conditions and params in parallel lists — is the correct pattern. The `?` placeholder keeps values out of the SQL string entirely, which prevents SQL injection and handles quoting correctly for any value type.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sqlite3
import pandas as pd


import sqlite3

def setup_db(conn):
    cur = conn.cursor()
    cur.execute('''
        CREATE TABLE IF NOT EXISTS orders (
            order_id  INTEGER PRIMARY KEY,
            product   TEXT,
            category  TEXT,
            region    TEXT,
            price     REAL,
            quantity  INTEGER,
            revenue   REAL
        )''')
    cur.execute('''
        CREATE TABLE IF NOT EXISTS products (
            product    TEXT PRIMARY KEY,
            category   TEXT,
            unit_price REAL
        )''')
    rows = [
        (1,'Widget','Electronics','North',25.0,10,250.0),
        (2,'Gadget','Electronics','South',150.0,3,450.0),
        (3,'Widget','Electronics','South',25.0,5,125.0),
        (4,'Doohickey','Accessories','East',8.0,50,400.0),
        (5,'Gadget','Electronics','East',150.0,7,1050.0),
        (6,'Widget','Electronics','East',25.0,4,100.0),
        (7,'Doohickey','Accessories','North',8.0,20,160.0),
        (8,'Gadget','Electronics','North',150.0,2,300.0),
        (9,'Widget','Electronics','West',25.0,6,150.0),
        (10,'Doohickey','Accessories','South',8.0,15,120.0),
        (11,'Thingamajig','Accessories','North',200.0,1,200.0),
        (12,'Thingamajig','Accessories','East',200.0,4,800.0),
    ]
    cur.executemany(
        'INSERT OR IGNORE INTO orders VALUES (?,?,?,?,?,?,?)', rows
    )
    products = [
        ('Widget','Electronics',25.0),
        ('Gadget','Electronics',150.0),
        ('Doohickey','Accessories',8.0),
        ('Thingamajig','Accessories',200.0),
    ]
    cur.executemany(
        'INSERT OR IGNORE INTO products VALUES (?,?,?)', products
    )
    conn.commit()


def run_query(conn, sql, params=()):
    cur = conn.cursor()
    cur.execute(sql, params)
    cols = [col[0] for col in cur.description]
    return [dict(zip(cols, row)) for row in cur.fetchall()]


conn = sqlite3.connect(':memory:')
setup_db(conn)

## Your Implementation

In [ ]:
def filter_orders(conn, region=None, category=None, min_revenue=None):
    """
    Query orders with optional WHERE filters.

    Build conditions and params in parallel lists.
    Join conditions with AND. Use run_query.

    Returns:
        list[dict] — matching rows ordered by order_id
    """
    conditions = []
    params = []
    # TODO: if region is not None: append 'region = ?' and region
    # TODO: if category is not None: append 'category = ?' and category
    # TODO: if min_revenue is not None: append 'revenue >= ?' and min_revenue
    # TODO: where = ('WHERE ' + ' AND '.join(conditions)) if conditions else ''
    # TODO: sql = f'SELECT * FROM orders {where} ORDER BY order_id'
    # TODO: return run_query(conn, sql, tuple(params))
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined
    try:
        assert 'filter_orders' in globals()
        passed += 1; print('\u2705 Check 1: filter_orders is defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: no filters returns all 12 rows
    try:
        all_rows = filter_orders(conn)
        assert len(all_rows) == 12, f'expected 12, got {len(all_rows)}'
        passed += 1; print('\u2705 Check 2: no filter returns all 12 rows')
    except Exception as e:
        print(f'\u274c Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: region='North' returns 4 rows
    try:
        north = filter_orders(conn, region='North')
        assert len(north) == 4, f'expected 4 North rows, got {len(north)}'
        assert all(r['region'] == 'North' for r in north)
        passed += 1; print('\u2705 Check 3: region=North returns 4 rows')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: category='Electronics' returns 7 rows
    try:
        elec = filter_orders(conn, category='Electronics')
        assert len(elec) == 7, f'expected 7 Electronics rows, got {len(elec)}'
        assert all(r['category'] == 'Electronics' for r in elec)
        passed += 1; print('\u2705 Check 4: category=Electronics returns 7 rows')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: combined region+category filter works
    try:
        east_elec = filter_orders(conn, region='East', category='Electronics')
        assert len(east_elec) == 2, \
            f'expected 2 East+Electronics rows, got {len(east_elec)}'
        assert all(r['region'] == 'East' and r['category'] == 'Electronics'
                   for r in east_elec)
        passed += 1; print('\u2705 Check 5: combined region+category filter works')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def filter_orders(conn, region=None, category=None, min_revenue=None):
    conditions = []
    params = []
    if region is not None:
        conditions.append('region = ?')
        params.append(region)
    if category is not None:
        conditions.append('category = ?')
        params.append(category)
    if min_revenue is not None:
        conditions.append('revenue >= ?')
        params.append(min_revenue)
    where = ('WHERE ' + ' AND '.join(conditions)) if conditions else ''
    sql = f'SELECT * FROM orders {where} ORDER BY order_id'
    return run_query(conn, sql, tuple(params))
```

</details>